# MIMIC-CXR v2 — RadHarmony Integration Example

This notebook demonstrates how to integrate [MIMIC-CXR v2.0](https://physionet.org/content/mimic-cxr/2.0.0/) into RadHarmony, following the same pattern described in `custom_dataset_tutorial.ipynb`.

---

## What this notebook covers

| Step | |
|---|---|
| 1 | `MIMICHarmonizer` — reads the MIMIC-CXR CSVs and returns a standardized DataFrame |
| 2 | `MIMICDataset` — PyTorch dataset wrapping the harmonizer |
| 3 | Validate the harmonized DataFrame |
| 4 | Split, inspect, and visualize |

---

## Dataset layout expected

```
mimic-cxr/2.1.0/
├── cxr-record-list.csv.gz      # per-image metadata (patient_id, study_id, dicom_id, path)
├── mimic-cxr-2.0.0-chexpert.csv  # CheXpert labels per study (optional, on NAS3)
├── cxr-study-list.csv.gz       # per-study report paths (optional)
└── files/                      ← base_image_dir points here
    └── p10/
        └── p10000032/
            └── s50414267/
                └── <dicom_id>.dcm / .jpg
```

---

## Step 1 — Harmonizer

`MIMICCXRHarmonizer` reads `cxr-record-list.csv.gz`, builds standardized `patient_id`, `study_id`, and `image_path` columns, snake-cases the label columns, and handles uncertain labels (`-1`).

Key decisions for MIMIC-CXR v2:
- `patient_id` and `study_id` are already present in the metadata CSV
- `image_path` comes directly from the `path` column (relative to dataset root)
- `view_position` is read from DICOM headers (`0x0018, 0x5101`) since MIMIC-CXR 2.0 has no `ViewPosition` column in the metadata CSV — requires `dicom_base_dir`
- Reports are loaded from `.txt` files listed in `cxr-study-list.csv.gz` — requires `report_csv_path` and `report_base_dir`
- Uncertain labels (`-1`) are converted to `NaN`; controlled by `drop_uncertain` (default: `True`)


In [ ]:
from radharmony.harmonizer import MIMICCXRHarmonizer

In [ ]:
mh = MIMICCXRHarmonizer(
    "/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/cxr-record-list.csv.gz",
    dicom_base_dir="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/files/",
    report_csv_path="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz",
    report_base_dir="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/files/",
)
df = mh.harmonize(drop_uncertain=False)
df

In [ ]:
df.view_position.value_counts(dropna=False)

---

## Step 2 — Dataset class

`MIMICCXRDataset` wraps `MIMICCXRHarmonizer` and declares the snake_cased label columns. Everything else — MONAI transforms, `PersistentDataset` caching, patient-level splitting — is inherited from `BaseRadiologicalDataset`.


In [ ]:
from radharmony.dataset import MIMICCXRDataset

---

## Step 3 — Validate the harmonized DataFrame

Check the harmonizer output before creating any datasets.

In [ ]:
import os

BASE_IMAGE_DIR = (
    "/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/files/"
)
CSV_PATH = "./temp_mimic.csv"
# CSV_PATH = "/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/cxr-record-list.csv.gz"
LABEL_CSV_PATH = "/path/to/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv"

df = MIMICCXRHarmonizer(
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    report_base_dir="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/files/",
    report_csv_path="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz",
).harmonize(drop_uncertain=True)

print(f"Shape          : {df.shape}")
print(f"Unique patients: {df['patient_id'].nunique():,}")
print(f"Unique studies : {df['study_id'].nunique():,}\n")

# Required columns
missing = [c for c in ["patient_id", "study_id", "image_path"] if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
print("Required columns: OK")

# Label range [0, 1]
label_cols_harmonized = [
    c.lower().replace(" ", "_") for c in MIMICCXRHarmonizer.LABEL_COLS
]
for col in label_cols_harmonized:
    if col in df.columns:
        assert df[col].between(0, 1).all(), f"Label '{col}' has values outside [0, 1]"
print(f"Label range [0,1]: OK  {label_cols_harmonized}")

# Spot-check image paths (5 samples)
for path in df.sample(min(5, len(df)), random_state=0)["image_path"]:
    full = os.path.join(BASE_IMAGE_DIR, path)
    assert os.path.exists(full), f"Image not found: {full}"
print("Image paths exist (5 sampled): OK\n")

# Label prevalence
print("Label prevalence:")
present = [c for c in label_cols_harmonized if c in df.columns]
print(df[present].mean().round(3).to_string())

df.head(3)

---

## Step 4 — Basic dataset usage

### 4.1 Train / validation split

`get_datasets(n_splits=N)` returns `(train_dataset, val_dataset)` with an ~`(N-1)/N` / `1/N` split. Splitting is **patient-level** — no patient appears in both sets.

In [ ]:
CACHE_DIR = "./cache/mimic"

ds = MIMICCXRDataset(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    report_csv_path="/path/to/MIMIC-CXR-V2/physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz",
    cache_dir=None,
    output_cls=True,
    output_report=True,
)

train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

### 4.2 Inspect a sample

Each sample is a dict. Keys present depend on the `output_*` flags enabled.

In [ ]:
import torch

sample = train_ds[0]

for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(
            f"  {key:12s}  shape={tuple(val.shape)}  dtype={val.dtype}  "
            f"min={val.float().min():.3f}  max={val.float().max():.3f}"
        )
    else:
        print(f"  {key:12s}  {type(val).__name__}: {val}")

In [ ]:
sample

### 4.3 Visualize a single sample

### 4.4 Visualize one sample per label

In [ ]:
import matplotlib.pyplot as plt

sample = train_ds[0]
img = sample["img"].float().squeeze().numpy()  # (H, W) for 2D

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
ax.axis("off")

if "cls" in sample:
    pos = [MIMICCXRDataset.LABEL_COLS[i] for i, v in enumerate(sample["cls"]) if v > 0]
    ax.set_title("  ".join(pos) if pos else "no_finding", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display

for fig in ds.visualize_samples(train_ds, per_label=True):
    display(fig)
    plt.close(fig)